## PyINE code variables analysis

This notebook performs an analysis of code snippets in the TACO dataset to determine the frequency and overlap of variable defintions they contain.

In [ ]:
import collections
import pathlib
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tqdm
import wordcloud

import pyine.data.taco.dataset_utils as taco_utils
import pyine.data.traces.dataset_utils as traces_utils
import pyine.utils.code.blocks
import pyine.utils.code.variables
import pyine.utils.filesystem
import pyine.utils.reprod

## Analysis

In [ ]:
# ------------ CHANGE THESE SETTINGS IF NEEDED ------------
source_dataset_name = "TACO"  # this notebook targets the TACO v1.3 10s10t traces dataset
source_dataset_path = pyine.data.taco.dataset_utils.get_latest_repackaged_dataset_path()
cluster_keyword_min_size = 3
cluster_min_keyword_freq = 100
cluster_max_keyword_freq = 1000
# ---------------------------------------------------------

stats_root_dir_path = pyine.utils.filesystem.get_data_root_path() / "stats"
stats_root_dir_path.mkdir(parents=True, exist_ok=True)
print(f"stats dump directory: {stats_root_dir_path}")
print("computing hashes for tagging/verification...")
source_dataset_hash = pyine.utils.reprod.compute_hash(source_dataset_path)
print(f" => source dataset full hash: {source_dataset_hash}")
stats_hash = pyine.utils.reprod.get_params_hash(
    source_dataset_name=source_dataset_name,
    source_dataset_path=source_dataset_path,
    source_dataset_hash=source_dataset_hash,
)
print(f" => stats hash: {stats_hash}")
cluster_data_hash = pyine.utils.reprod.get_params_hash(
    stats_hash=stats_hash,
    cluster_keyword_min_size=cluster_keyword_min_size,
    cluster_min_keyword_freq=cluster_keyword_min_size,
    cluster_max_keyword_freq=cluster_max_keyword_freq,
)
print(f" => cluster data hash: {cluster_data_hash}")

CODE_BLOCK_STATS_FILE_NAME = stats_root_dir_path / f"code_block_stats.{stats_hash[:16]}.pickle"
VARIABLE_METHOD_STATS_FILE_NAME = stats_root_dir_path / f"variable_and_method_stats.{stats_hash[:16]}.pickle"
DEFINITION_CLUSTER_STATS_FILE_NAME = stats_root_dir_path / f"definition_clusters.{cluster_data_hash[:16]}.pickle"

In [ ]:
def apply_function_to_code_snippets(func_to_apply):
    count = 0
    errors = 0
    result = {}
    problem_iterator = traces_utils.CodingProblemIterator(
        dataset_name=source_dataset_name,
        root_data_path=source_dataset_path,
        reformat_code_strings=False,
        validate_code_strings=False,
        enable_async_prefetch=True,
        show_progress=True,
    )
    for problem, solutions in problem_iterator:
        if problem.should_discard():
            continue
        for solution in solutions:
            if solution.should_discard():
                continue
            try:
                current_result = func_to_apply(solution.code)
                result[str(solution.solution_id)] = current_result
                count += 1
            except SyntaxError:
                errors += 1
    print(f"found {errors} code snippets with errors")
    return result

In [ ]:
if pathlib.Path(CODE_BLOCK_STATS_FILE_NAME).is_file():
    print(f"stats already there for {CODE_BLOCK_STATS_FILE_NAME} - not computing again")
else:
    code_block_stats = apply_function_to_code_snippets(pyine.utils.code.blocks.identify_code_blocks)
    with open(CODE_BLOCK_STATS_FILE_NAME, "wb") as f:
        pickle.dump(code_block_stats, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if pathlib.Path(VARIABLE_METHOD_STATS_FILE_NAME).is_file():
    print(f"stats already there for {VARIABLE_METHOD_STATS_FILE_NAME} - not computing again")
else:
    variable_method_stats = apply_function_to_code_snippets(pyine.utils.code.variables.analyze_definitions)
    with open(VARIABLE_METHOD_STATS_FILE_NAME, "wb") as f:
        pickle.dump(variable_method_stats, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
if pathlib.Path(DEFINITION_CLUSTER_STATS_FILE_NAME).is_file():
    print(f"clusters already there for {DEFINITION_CLUSTER_STATS_FILE_NAME} - not computing again")
    with open(DEFINITION_CLUSTER_STATS_FILE_NAME, "rb") as f:
        definition_cluster_payload = pickle.load(f)
else:
    print("building definition clusters from dataset...")
    cluster_codes: list[str] = []
    cluster_solution_ids: list[str] = []
    problem_iterator = traces_utils.CodingProblemIterator(
        dataset_name=source_dataset_name,
        root_data_path=source_dataset_path,
        reformat_code_strings=False,
        validate_code_strings=False,
        enable_async_prefetch=True,
        show_progress=True,
    )
    for problem, solutions in tqdm.tqdm(problem_iterator, desc="collecting snippets for clustering"):
        if problem.should_discard():
            continue
        for solution in solutions:
            if solution.should_discard():
                continue
            cluster_solution_ids.append(str(solution.solution_id))
            cluster_codes.append(solution.code)
    definition_clusters = pyine.utils.code.variables.cluster_code_snippets_by_keyword(
        cluster_codes,
        min_keyword_frequency=cluster_min_keyword_freq,
        max_keyword_frequency=cluster_max_keyword_freq,
        min_keyword_length=cluster_keyword_min_size,
        banned_keywords={"self", "cls"},
        keyword_transform=str.lower,  # noqa
        raise_on_error=False,
        verbose=True,
    )
    definition_cluster_payload = {
        "clusters": definition_clusters,
        "solution_ids": cluster_solution_ids,
    }
    with open(DEFINITION_CLUSTER_STATS_FILE_NAME, "wb") as f:
        pickle.dump(definition_cluster_payload, f, protocol=pickle.HIGHEST_PROTOCOL)
definition_clusters = definition_cluster_payload["clusters"]
definition_cluster_solution_ids = definition_cluster_payload["solution_ids"]
print(f"{len(definition_clusters)=:,}")

## Plotting info on all code variables

In [ ]:
with open(CODE_BLOCK_STATS_FILE_NAME, "rb") as f:
    code_block_stats = pickle.load(f)
print(f"{len(code_block_stats)=:,}")

with open(VARIABLE_METHOD_STATS_FILE_NAME, "rb") as f:
    variable_method_stats = pickle.load(f)
print(f"{len(variable_method_stats)=:,}")

keywords = [kw for d in variable_method_stats.values() for kw in d.get("bound_names", set())]


def norm(s: str) -> str:
    return s.strip().lower()


counts = collections.Counter(norm(k) for k in keywords if k.strip())
wc = wordcloud.WordCloud(width=1200, height=600, background_color="white")
wc = wc.generate_from_frequencies(counts)

plt.figure(figsize=(12, 6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
counts = collections.Counter(kw for d in variable_method_stats.values() for kw in d.get("bound_names", set()))
freqs = np.fromiter(counts.values(), dtype=int)

plt.figure(figsize=(10, 4))
plt.hist(freqs, bins="auto", log=True)  # log=True => y-axis log scale
plt.xscale("log")  # x-axis log scale
plt.xlabel("Frequency (how many top-level items contain the keyword) (log scale)")
plt.ylabel("Number of keywords (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
N = 50  # show only the most frequent N
counts = collections.Counter(kw for d in variable_method_stats.values() for kw in d.get("bound_names", set()))
top_items = counts.most_common(N)
labels = [k for k, _ in top_items]
freqs = [v for _, v in top_items]
x = np.arange(len(freqs))

plt.figure(figsize=(12, 5))
plt.bar(x, freqs)
plt.xticks(x, labels, rotation=45, ha="right", rotation_mode="anchor")
plt.xlabel("Keyword")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## Plotting/displaying info on keyword clusters


In [ ]:
definition_cluster_records = [
    {
        "keyword": cluster.keyword,
        "snippet_count": len(cluster.code_snippet_indices),
        "keyword_length": len(cluster.keyword),
    }
    for cluster in definition_clusters
]
definition_cluster_df = (
    pd.DataFrame(definition_cluster_records).sort_values("snippet_count", ascending=False).reset_index(drop=True)
)
definition_cluster_df

In [ ]:
clustered_indices: set[int] = set()
for cluster in definition_clusters:
    clustered_indices.update(cluster.code_snippet_indices)
total_snippet_count = len(definition_cluster_solution_ids)
cluster_coverage_ratio = len(clustered_indices) / total_snippet_count if total_snippet_count else np.nan
definition_cluster_stats_df = pd.DataFrame(
    [
        {"metric": "cluster_count", "value": len(definition_clusters)},
        {"metric": "total_snippet_count", "value": total_snippet_count},
        {"metric": "clustered_snippet_count", "value": len(clustered_indices)},
        {"metric": "clustered_snippet_ratio", "value": cluster_coverage_ratio},
        {"metric": "median_cluster_size", "value": definition_cluster_df["snippet_count"].median()},
        {"metric": "mean_cluster_size", "value": definition_cluster_df["snippet_count"].mean()},
    ]
)
definition_cluster_stats_df

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(definition_cluster_df["snippet_count"], bins="auto", log=True)
plt.xscale("log")
plt.xlabel("Snippets per keyword (log scale)")
plt.ylabel("Number of keywords (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
TOP_CLUSTER_COUNT = 80
top_cluster_subset = definition_cluster_df.head(TOP_CLUSTER_COUNT)
plt.figure(figsize=(12, 5))
plt.bar(top_cluster_subset.index, top_cluster_subset["snippet_count"])
plt.xticks(
    top_cluster_subset.index,
    top_cluster_subset["keyword"],
    rotation=70,
    ha="right",
    rotation_mode="anchor",
)
plt.xlabel("Keyword")
plt.ylabel("Snippet count")
plt.tight_layout()
plt.show()

In [ ]:
example_rows = []
for cluster in definition_clusters:
    sample_solution_ids = [definition_cluster_solution_ids[idx] for idx in cluster.code_snippet_indices]
    example_rows.append(
        {
            "keyword": cluster.keyword,
            "snippet_count": len(cluster.code_snippet_indices),
            "sample_solution_ids": ", ".join(sample_solution_ids),
        }
    )
definition_cluster_examples_df = pd.DataFrame(example_rows)
definition_cluster_examples_df